![Who plays devil's advocate? — NucleusIQ investment board showcase](nucleusiq-devils-advocate-newsletter-cover.png)

*Cover image for LinkedIn / newsletter — same folder as this notebook.*

---

# NucleusIQ Framework Showcase — Investment Board Devil's Advocate

This notebook is a **product demonstration** of the **NucleusIQ Agent framework**.  
It is not a generic chatbot demo — every phase is designed to show **why** you would use NucleusIQ in production and **which framework capabilities** each step exercises.

**Provider:** Google Gemini via `nucleusiq-gemini`  
**Domain:** Investment committee chair needs a **devil's advocate** before approving **$500K–$1M** in Indian IT equities (**TCS** vs **HCL**).

> **Disclaimer:** Educational demo only — not investment, legal, or tax advice.

---

## Why we are building this

### The business problem

A **board chair** must decide whether to deploy **$500K–$1M** across **TCS** and/or **HCL**, using:

- FY2025 **annual report** financials (public data)
- A **credit memo** (cautious: approve $600K, 60/40 TCS/HCL)
- An **RM memo** (bullish: approve $900K, 45/55 TCS/HCL)
- **Five analyst** board memos (split buy / hold / sell)
- Email-style **commentary** between credit and RM

The chair cannot read 300+ page PDFs and five memos under time pressure. They need a **repeatable skeptical partner** that:

1. Has read the **frozen pack** before the meeting  
2. Cites **section_id** evidence (not invented numbers)  
3. Surfaces **credit vs RM vs analyst** disagreements  
4. Answers **follow-up** questions in the room (“What if we only do $600K?”)

### Why NucleusIQ (not a raw LLM script)

| Need | Raw LLM | **NucleusIQ** |
|------|---------|----------------|
| Read structured pack on demand | Paste entire JSON every turn | **`@tool`** — `get_pack_section`, `summarize_credit_and_rm` |
| Multi-step reasoning | Fragile prompt chains | **`Agent`** + **STANDARD** tool loop |
| Fast chair Q&A | Same heavy prompt | **`ExecutionMode.DIRECT`** |
| Conversation memory | Manual history string | **`MemoryFactory`** (sliding window, summary, …) |
| Tool failures | Custom retry while-loops | **`ToolRetryPlugin`** |
| Runaway cost / loops | Hope | **`ModelCallLimitPlugin`**, **`ToolCallLimitPlugin`** |
| Observability | Opaque | **`agent.last_usage`**, **`CostTracker`** |

**This notebook proves the framework** on a realistic committee workflow — data prep is custom; **the agent layer is 100% NucleusIQ**.

### What we deliberately do NOT use

- Bank CRM exports (`reallocation.json` / `renewal.json`) — removed; we build a **custom canonical pack**
- NucleusIQ core does **not** own PDF ETL — Phase A is a small custom pipeline, then the **Agent consumes JSON**

---

## NucleusIQ features demonstrated (checklist)

| # | Framework feature | Where in this notebook |
|---|-------------------|------------------------|
| 1 | **`Agent`** orchestrator | Phase C, Phase D |
| 2 | **`ExecutionMode.STANDARD`** | Phase C — preload / tool loop |
| 3 | **`ExecutionMode.DIRECT`** | Phase D — board chat |
| 4 | **`ZeroShotPrompt`** | System + user prompts in `committee/showcase_agent.py` |
| 5 | **`@tool` decorator** | Pack tools: `get_pack_section`, `search_pack`, … |
| 6 | **`Task`** | Each chair question = one task |
| 7 | **`ModelCallLimitPlugin`** | Safety ceiling on LLM calls |
| 8 | **`ToolCallLimitPlugin`** | Safety ceiling on tool calls |
| 9 | **`ToolRetryPlugin`** | Retries — **no hand-written retry loops** |
| 10 | **`MemoryFactory`** | Phase D — `sliding_window`, `summary`, … |
| 11 | **`CostTracker`** + **`last_usage`** | Printed after each reply |
| 12 | **`Agent.initialize()`** | Before execute |
| 13 | **Structured output (Pydantic)** | Optional objection register on preload |

Compare with [`research_analyst_tcs.ipynb`](../../research_analyst_tcs.ipynb) for PDF/web research patterns.

## Architecture (what this demo proves)

```text
  [FY2025 PDFs: TCS + HCL]
           │
           ▼
  ┌────────────────────────────┐
  │ Phase A — Custom data prep │  ← NOT NucleusIQ core (your ETL)
  │  → investment_committee_   │
  │    pack.json               │
  └─────────────┬──────────────┘
                │
                ▼
  ┌────────────────────────────┐
  │ Phase C — NucleusIQ Agent  │  STANDARD mode + @tools + plugins
  │  T−1 chair pre-brief       │
  └─────────────┬──────────────┘
                │
                ▼
  ┌────────────────────────────┐
  │ Phase D — NucleusIQ Agent  │  DIRECT mode + MemoryFactory
  │  Multi-turn devil's advocate│  + ToolRetryPlugin
  └────────────────────────────┘
```

**Intentional tension in the pack** (so devil's advocate has material):

| Voice | Recommendation | Amount |
|-------|----------------|--------|
| Credit | APPROVE_WITH_CONDITIONS | $600K |
| RM | APPROVE | $900K |
| Analysts | Mixed buy / hold / sell | $250K–$750K |

---

## Run guide — execute cells in order

| Step | Phase | API key? | Time |
|------|-------|----------|------|
| 1 | **Setup** | No | 1 min |
| 2 | **A — Build pack** | No | ~1 min |
| 3 | **B — Review pack** | No | 2 min |
| 4 | **C — Preload** | **Yes** `GEMINI_API_KEY` | 1–3 min |
| 5 | **D — Chat** | **Yes** | per question |

**Prerequisites**

- PDFs at repo root: `research/tcs_annual_report-2025.pdf`, `research/hcl_2025_Financial_report.pdf`
- `GEMINI_API_KEY` in `NucleusIQ/.env` for phases C & D

**Phase D tips**

- Run **Session ready** once, then example questions, then the **interactive chat** cell
- Type `reset` to clear memory; `quit` to exit
- Change `MEMORY_STRATEGY` to showcase `summary` or `summary_window`

---
## Setup

Install deps, point Python at `committee/`, load `.env`. The cell after `%pip` prints paths and **confirms NucleusIQ imports** for phases C & D.

In [1]:
%pip install -q nucleusiq nucleusiq-gemini pypdf python-dotenv pandas jsonschema

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

# Locate package root (parent of notebooks/)
NB_DIR = Path.cwd().resolve()
if (NB_DIR / "committee").is_dir():
    PKG = NB_DIR.parent
elif (NB_DIR.parent / "committee").is_dir():
    PKG = NB_DIR.parent
else:
    PKG = NB_DIR
REPO = PKG.parent.parent.parent
sys.path.insert(0, str(PKG))

from committee.env import load_env
from committee.paths import (
    DISCLAIMER,
    HCL_2025_PDF,
    OBJECTIONS_OUTPUT,
    PACK_OUTPUT,
    PREBRIEF_OUTPUT,
    TCS_2025_PDF,
)
from committee.pack_builder import write_pack
from committee.showcase_agent import MEMORY_CHOICES, ChairChatSession, run_preload

load_env()
print("Repo:", REPO)
print("Package:", PKG)
print("GEMINI_API_KEY set:", bool(os.environ.get("GEMINI_API_KEY")))
print("PDFs:", TCS_2025_PDF.is_file(), HCL_2025_PDF.is_file())

Repo: C:\Users\Brijesh Singh\Documents\github-project\NucleusIQ
Package: C:\Users\Brijesh Singh\Documents\github-project\NucleusIQ\notebooks\agents\investment_board_devils_advocate
GEMINI_API_KEY set: True
PDFs: True True


---
## Phase A — Data preparation (build pack)

### Why this phase exists

The **Agent must not invent financials**. Phase A produces a **frozen, citeable JSON pack** the chair would trust — same pattern as a bank “board pack,” but built from **public annual reports** for this showcase.

### What gets built (custom pipeline — not NucleusIQ core)

| Block | Purpose |
|-------|---------|
| `issuers[].financials` | FY2025 P&L + BS lines from PDF (deterministic extract) |
| `credit_analysis` | Ratios, risk rollup, conditions |
| `credit_memo` | Cautious credit view |
| `rm_memo` | Bullish RM view (disagrees with credit) |
| `commentary_thread` | Credit ↔ RM debate |
| `analyst_memos[]` | Five board analysts |

**Output file:** `data/02_output/investment_committee_pack.json`

> NucleusIQ starts in **Phase C** — the framework consumes this JSON via `@tool` functions.

In [3]:
assert TCS_2025_PDF.is_file(), f"Missing {TCS_2025_PDF}"
assert HCL_2025_PDF.is_file(), f"Missing {HCL_2025_PDF}"

pack = write_pack()
print("Wrote:", PACK_OUTPUT)
print("Version:", pack["meta"].get("pack_version"))
print(DISCLAIMER)

Wrote: C:\Users\Brijesh Singh\Documents\github-project\NucleusIQ\notebooks\agents\investment_board_devils_advocate\data\02_output\investment_committee_pack.json
Version: 0.2.0
Educational demonstration only. Not investment, legal, or tax advice. Figures are taken from cited annual report excerpts; verify independently before any decision.


In [4]:
# FY2025 comparison table
pd.DataFrame(pack["comparison"]["rows"])

,issuer_id,company_name,revenue_inr_crore,revenue_yoy_pct,net_margin_pct,pat_inr_crore
0,tcs,Tata Consultancy Services Limited,255324.0,5.99,19.11,48797.0
1,hcl,HCL Technologies Limited,117055.0,6.50,14.86,17399.0


---
## Phase B — Review pack (human sign-off before Agent)

### Why this phase exists

You are the **product owner / chair** for this demo. Skim the same artifacts the Agent will query via `@tool` — if credit/RM tension looks wrong here, fix `committee/credit_artifacts.py` and re-run Phase A.

**Do not skip** — last checkpoint before spending API tokens in phases C & D.

In [5]:
cm = pack["credit_memo"]
rm = pack["rm_memo"]
print("CREDIT:", cm["recommendation"], f"${cm['proposed_allocation_usd']:,}")
print("RM:   ", rm["recommendation"], f"${rm['proposed_allocation_usd']:,}", rm["proposed_split"])
print("\n--- Credit memo (executive summary) ---\n")
print(next(s["text"] for s in cm["sections"] if "executive" in s["section_id"]))
print("\n--- RM memo (recommendation) ---\n")
print(next(s["text"] for s in rm["sections"] if s["section_id"].endswith("recommendation")))

CREDIT: APPROVE_WITH_CONDITIONS $600,000
RM:    APPROVE $900,000 {'tcs': 0.45, 'hcl': 0.55}

--- Credit memo (executive summary) ---

Credit recommends **approve with conditions** a **$600,000** deployment (60% TCS / 40% HCL), not the full $1M band. FY25 consolidated data shows TCS revenue ₹255,324 cr (margin 19.11%) vs HCL ₹117,055 cr (margin 14.86%). Quality is acceptable; **timing and sizing** are the debate.

--- RM memo (recommendation) ---

**Approve $900K** — 55% HCL / 45% TCS. Relationship strategy: **Grow**. Differs from credit on size (+$300K) and HCL overweight (+15 pts vs credit 60/40 TCS-heavy).


In [6]:
print("--- Commentary thread ---")
for c in pack["commentary_thread"]:
    print(f"[{c['date']}] {c['role']}: {c['text']}")

--- Commentary thread ---
[2026-05-18] Credit: We should not bring $1M to board without live multiples. Proposing $600K 60/40 TCS/HCL.
[2026-05-18] RM: Client will view $600K as under-deployment. $900K 45/55 TCS/HCL aligns with mandate and peer benchmarks.
[2026-05-19] Credit: HCL cash down YoY in FY25 extract — RM please attach WC bridge or we keep AMBER rollup.
[2026-05-19] RM: Cash dip is timing. TCS margin gap vs HCL is structural quality — don't underweight TCS below 40%.
[2026-05-20] Credit: Board pack must show both views. I remain APPROVE_WITH_CONDITIONS; RM remains full APPROVE at $900K.
[2026-05-20] RM: Chair: if you want devil's advocate, ask why credit wants to leave $400K uninvested in a 6% revenue growth sector.
[2026-05-21] Secretariat: Pack frozen. Credit memo, RM memo, FY25 extracts, five analyst views included. Chair decision required.


In [7]:
print("--- Analyst stances ---")
for m in pack["analyst_memos"]:
    print(f"{m['analyst_id']}: {m['stance'].upper()} ${m['target_position_usd']:,} — {m['display_name']}")

--- Analyst stances ---
A: BUY $750,000 — Analyst A — TCS quality
B: BUY $650,000 — Analyst B — HCL growth
C: HOLD $500,000 — Analyst C — Balanced barbell
D: SELL $0 — Analyst D — Valuation skeptic
E: SELL $250,000 — Analyst E — Macro underweight


---
## Phase C — T−1 preload (NucleusIQ **STANDARD** mode)

### Why this phase exists

The night before the board meeting, the chair wants a **written challenge brief** — objections, questions, credit vs RM gaps — without running the live chat yet.

### Framework power on display

| Capability | How you see it |
|------------|----------------|
| **`Agent`** | One orchestrator runs the full analysis |
| **`ExecutionMode.STANDARD`** | Multi-step **tool loop** (not one-shot prompt) |
| **`@tool`** | Agent calls `summarize_credit_and_rm`, `get_pack_section`, … |
| **Plugins** | Limits + **`ToolRetryPlugin`** (framework handles retries) |
| **`Task`** | Single objective: produce devil's advocate pre-brief |

This is the same pattern as production “run analysis job overnight” — not a chat UI.

In [8]:
assert os.environ.get("GEMINI_API_KEY"), "Set GEMINI_API_KEY in repo .env"

preload_result = await run_preload(PACK_OUTPUT)
preload_result

[INFO] devils-advocate-preload: Initializing agent: devils-advocate-preload
[INFO] devils-advocate-preload: Agent initialization completed successfully
[INFO] devils-advocate-preload: Agent 'devils-advocate-preload' executing in STANDARD mode
[INFO] devils-advocate-preload: Tool requested: summarize_credit_and_rm
[INFO] devils-advocate-preload: Tool requested: list_commentary_thread


{'status': 'ResultStatus.SUCCESS',
 'prebrief_path': 'C:\\Users\\Brijesh Singh\\Documents\\github-project\\NucleusIQ\\notebooks\\agents\\investment_board_devils_advocate\\data\\02_output\\chair_prebrief.md',
 'usage': 'tokens in=10117 out=1451 | est. $0.0012',
 'framework': {'mode': 'STANDARD',
  'plugins': ['ModelCallLimit', 'ToolCallLimit', 'ToolRetry'],
  'memory': None}}

In [9]:
if PREBRIEF_OUTPUT.is_file():
    print(PREBRIEF_OUTPUT.read_text(encoding="utf-8")[:5000])

I am not convinced this deal is ready for approval. The presented information reveals significant discrepancies, unsupported claims, and critical information gaps that warrant a strong NO vote at this time.

### Objections to the Proposed Deal:

1.  **Egregious Credit vs. RM Mismatch on Amounts and Conditions:** The core recommendation is fundamentally fractured. Credit proposes a cautious **$600,000 with conditions** and a "Watchlist-lite" grade (section_id: `credit_memo.root`, `credit_memo.recommendation`), while RM pushes for a significantly higher **$900,000** with an outright "APPROVE" and "Grow" strategy (section_id: `rm_memo.root`, `rm_memo.recommendation`). This $300,000 difference, representing a 50% increase over Credit's recommendation, is not a minor disagreement; it signals a fundamental divergence in risk assessment and investment appetite. The Credit memo's "APPROVE_WITH_CONDITIONS" (section_id: `credit_memo.recommendation`) is effectively a red flag, indicating that the

---
## Phase D — Board meeting chat (NucleusIQ **DIRECT** + memory)

### Why this phase exists

During the meeting the **chair** asks spontaneous questions:  
*“RM wants $900K — why not?”* → *“Which section shows cash declining?”* → *“Strongest reason to pass?”*

A one-shot LLM cannot do this well. NucleusIQ provides:

| Capability | Why it matters for the chair |
|------------|------------------------------|
| **`ExecutionMode.DIRECT`** | Faster responses in live Q&A |
| **`MemoryFactory`** | Follow-ups reference prior answers |
| **`@tool`** | Grounding in pack on every turn |
| **`ToolRetryPlugin`** | Resilient tool calls under load |
| **`CostTracker`** | Transparent token/cost per turn |

**This is the hero demo** — open the interactive cell and act as the chair.

Set memory strategy (re-run **Session ready** after changing):

In [10]:
MEMORY_STRATEGY = "sliding_window"  # or: summary, summary_window, full_history, token_budget
os.environ["MEMORY_STRATEGY"] = MEMORY_STRATEGY
print("Available:", list(MEMORY_CHOICES.keys()))
print("Using:", MEMORY_STRATEGY)

Available: ['sliding_window', 'summary', 'summary_window', 'full_history', 'token_budget']
Using: sliding_window


In [11]:
assert os.environ.get("GEMINI_API_KEY"), "Set GEMINI_API_KEY"

chair = ChairChatSession(PACK_OUTPUT, memory_strategy=MEMORY_STRATEGY)
await chair.start()
print("Session ready.")
print(json.dumps(chair.framework_info, indent=2))

[INFO] devils-advocate-chat: Initializing agent: devils-advocate-chat
[INFO] devils-advocate-chat: Agent initialization completed successfully


Session ready.
{
  "execution_mode": "DIRECT",
  "memory": "sliding_window",
  "memory_class": "SlidingWindowMemory",
  "plugins": "ModelCallLimit, ToolCallLimit, ToolRetry"
}


### Example questions (run one at a time)

In [12]:
q = (
    "RM wants $900K with 55% HCL. Credit only wants $600K at 60% TCS. "
    "Give me the three strongest reasons to REJECT the RM plan today. Cite section_ids."
)
print("You:", q)
reply, usage = await chair.ask(q)
print("\nDevil's advocate:\n", reply)
print("\n", usage)

[INFO] devils-advocate-chat: Agent 'devils-advocate-chat' executing in DIRECT mode


You: RM wants $900K with 55% HCL. Credit only wants $600K at 60% TCS. Give me the three strongest reasons to REJECT the RM plan today. Cite section_ids.

Devil's advocate:
 Here are three strong reasons to reject the RM plan today:

1.  **Fundamental Disagreement on Deal Size and Conditions:** The RM plan for $900,000 is a significant overreach compared to Credit's cautious recommendation of $600,000 (section_id: `rm_memo.root`, `credit_memo.root`). Credit explicitly states "APPROVE_WITH_CONDITIONS" and assigns a "Watchlist-lite" grade (section_id: `credit_memo.recommendation`), indicating that the deal, as presented by RM, does not meet their full approval criteria and carries unmitigated risks. The $300,000 difference represents a 50% increase over Credit's proposed amount, highlighting a severe divergence in risk assessment that has not been reconciled.

2.  **RM's HCL Bias Contradicts Financial Performance:** The RM plan advocates for a 55% allocation to HCL (section_id: `rm_memo.r

In [13]:
q2 = "Which objection matters most if we only approve $600K credit split?"
print("You:", q2)
reply2, usage2 = await chair.ask(q2)
print("\nDevil's advocate:\n", reply2)
print("\n", usage2)

[INFO] devils-advocate-chat: Agent 'devils-advocate-chat' executing in DIRECT mode


You: Which objection matters most if we only approve $600K credit split?

Devil's advocate:
 Here are the three strongest reasons to REJECT the RM plan today:

1.  **Fundamental Discrepancy in Risk Assessment and Proposed Allocation:** The RM's recommendation for a $900,000 deployment with a 55% allocation to HCL (section_id: `rm_memo.root`) is fundamentally at odds with Credit's cautious $600,000 recommendation, which initially proposed a 60% allocation to TCS (section_id: `credit_memo.root`, `commentary_thread.001`). This is not a minor disagreement; it represents a 50% increase in the proposed investment amount and a significant divergence in risk appetite and asset preference. Credit's "APPROVE_WITH_CONDITIONS" (section_id: `credit_memo.recommendation`) explicitly indicates that the deal, as presented by RM, does not meet their full approval criteria, making the RM plan premature and inadequately supported.

2.  **RM's HCL Preference Contradicted by Financials and Unresolved Risks:

### Interactive chat loop — act as the chair

Run the cell below. This is the **live framework showcase**:

1. You type as **chair** at `You >`
2. **NucleusIQ Agent** (DIRECT + memory + tools + plugins) responds as **devil's advocate**
3. Token/cost line prints after each turn (`CostTracker`)

Commands: **`quit`** exit | **`reset`** new meeting

**Demo script for stakeholders**

1. “RM wants $900K 55% HCL — convince me not to approve.”
2. “What does credit say about cash vs RM narrative?”
3. “If we only approve $600K — what do we still not know?” (follow-up — proves **memory**)

In [ ]:
print("Interactive chat — type quit to exit, reset to clear memory\n")
print("Framework:", chair.framework_info)

while True:
    try:
        user_input = input("\nYou > ").strip()
    except KeyboardInterrupt:
        print("\nStopped.")
        break
    if not user_input:
        continue
    low = user_input.lower()
    if low in {"quit", "exit", "q"}:
        print("Goodbye.")
        break
    if low == "reset":
        chair.reset_memory()
        print("[Memory cleared]\n")
        continue
    print("\n... thinking ...\n")
    reply, usage = await chair.ask(user_input)
    print("\nDevil's advocate:\n", reply)
    print("\n", usage)

Interactive chat — type quit to exit, reset to clear memory

Framework: {'execution_mode': 'DIRECT', 'memory': 'sliding_window', 'memory_class': 'SlidingWindowMemory', 'plugins': 'ModelCallLimit, ToolCallLimit, ToolRetry'}


---
## Appendix — edit custom content

To change credit/RM narrative before rebuilding:

| File | Content |
|------|---------|
| `committee/credit_artifacts.py` | credit_memo, rm_memo, commentary, credit_analysis |
| `committee/memos.py` | five analyst memos |
| `committee/pdf_financials.py` | PDF page extraction |

Re-run **Phase A** after edits.